# 03 — Feature engineering

In [1]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score

from fraud.data import load_train
from fraud.split import time_split

In [2]:
df = load_train()
train, valid, _ = time_split(df)

In [3]:
NON_FEATURES = ["isFraud", "TransactionDT", "TransactionID"]

MODEL_PARAMS = dict(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=63,
    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.5,
    n_jobs=8,
    verbose=-1,
)

def evaluate(train, valid, seed=42):
    """Fit on train, score on valid."""
    model = lgb.LGBMClassifier(**MODEL_PARAMS,random_state=seed)
    model.fit(train.drop(columns=NON_FEATURES), train['isFraud'])
    scores = model.predict_proba(valid.drop(columns=NON_FEATURES))[:, 1]
    roc = roc_auc_score(valid["isFraud"], scores)
    pr = average_precision_score(valid["isFraud"], scores)
    print(f"seed {seed:<3} ROC-AUC {roc:.4f}   PR-AUC {pr:.4f}")

evaluate(train, valid)

seed 42  ROC-AUC 0.9247   PR-AUC 0.5884


In [4]:
for seed in [1, 2]:
    evaluate(train, valid, seed)

seed 1   ROC-AUC 0.9213   PR-AUC 0.5836


seed 2   ROC-AUC 0.9254   PR-AUC 0.5941


## 1. Time features

In [5]:
from fraud.features import add_time_features

df = add_time_features(df)
train, valid, _ = time_split(df)

In [6]:
(train.groupby("hour")["isFraud"].mean() * 100).round(1)

hour
0      3.2
1      3.3
2      3.6
3      3.5
4      5.0
5      6.2
6      7.6
7      9.7
8     10.0
9      9.6
10     7.5
11     4.1
12     3.1
13     2.2
14     2.4
15     2.5
16     3.0
17     3.2
18     3.5
19     3.5
20     3.4
21     3.4
22     3.3
23     3.9
Name: isFraud, dtype: float64

In [7]:
(train.groupby("weekday")["isFraud"].mean() * 100).round(1)

weekday
0    3.8
1    3.6
2    3.7
3    3.7
4    3.2
5    3.3
6    3.4
Name: isFraud, dtype: float64

In [8]:
for seed in [42, 1, 2]:
    evaluate(train, valid, seed)

seed 42  ROC-AUC 0.9222   PR-AUC 0.5835


seed 1   ROC-AUC 0.9222   PR-AUC 0.5907


seed 2   ROC-AUC 0.9220   PR-AUC 0.5891


`hour` and `weekday` are computed from a single row, so nothing is learned from other rows and they cannot leak.
The fraud rate differs strongly by hour on train (2.2% to 10.0%), but adding both features does not improve the
model: mean PR-AUC over three seeds is 0.589 without them and 0.588 with them, well within the seed noise of about
0.01. They stay out of the main model and will be re-checked for logistic regression and the lite demo model.